In [1]:
import os
os.chdir("../")
%pwd

'c:\\MySpace\\computer language\\Python\\MLOPS\\MLOPS by krish naik\\Projects\\datascienceproject'

In [2]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/pervnandal/datascienceproject.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="pervnandal"
os.environ["MLFLOW_TRACKING_PASSWORD"]="fa5b1231b15d1f28616c8c79c9af5c67fab087e3"

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params:dict
    metric_file_name:Path
    target_column:str
    mlflow_uri:str

In [4]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml,create_directories,save_json

In [9]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self)->ModelEvaluationConfig:
        config=self.config.model_evaluation
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN
        
        create_directories([config.root_dir])
        
        model_evaluation_config=ModelEvaluationConfig(
            root_dir= config.root_dir,
            test_data_path= config.test_data_path,
            model_path= config.model_path,
            all_params= params,
            metric_file_name= config.metric_file_name,
            target_column= schema,
            mlflow_uri= os.environ["MLFLOW_TRACKING_URI"]
        )
        return model_evaluation_config

In [6]:
import os
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

c:\MySpace\computer language\Python\MLOPS\MLOPS by krish naik\Projects\datascienceproject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class ModelEvaluation:
    def __init__(self,config: ModelEvaluationConfig):
        self.config=config
    
    def eval_metrics(self,actual,pred):
        rmse=np.sqrt(mean_squared_error(actual,pred))
        mae=mean_absolute_error(actual,pred)
        r2=r2_score(actual,pred)
        
        return rmse,mae,r2
    
    def log_mlflow(self):
        
        test_data=pd.read_csv(self.config.test_data_path)
        model=joblib.load(self.config.model_path)
        
        test_x=test_data.drop([self.config.target_column],axis=1)
        test_y=test_data[self.config.target_column]
        
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store=urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            predicted_qualities=model.predict(test_x)
            
            (rmse,mae,r2)=self.eval_metrics(test_y,predicted_qualities)
            
            # saving metrics as local
            scores={"rmse":rmse,"mae":mae,"r2":r2}
            save_json(path=Path(self.config.metric_file_name),data=scores)
            
            mlflow.log_params(self.config.all_params)
            
            mlflow.log_metric("rmse",rmse)
            mlflow.log_metric("mae",mae)
            mlflow.log_metric("r2",r2)
            
            # Model rehistery does not work with file store
            if tracking_url_type_store != "file":
            
                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow

                mlflow.sklearn.log_model(sk_model=model,
                                         name="model",
                                         registered_model_name="ElasticNetModel")
            else:
                mlflow.sklearn.log_model(sk_model=model,name="model")

In [10]:
try:
    config = ConfigurationManager()
    model_evaluation_config=config.get_model_evaluation_config()
    model_evaluation=ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_mlflow()
except Exception as e:
    raise e

[2026-09-28 11:48:24,969: INFO: common: Yaml file: config\config.yaml loaded successfully]
[2026-09-28 11:48:24,971: INFO: common: Yaml file: params.yaml loaded successfully]
[2026-09-28 11:48:24,973: INFO: common: Yaml file: schema.yaml loaded successfully]
[2026-09-28 11:48:24,974: INFO: common: created directory at: artifacts]
[2026-09-28 11:48:24,975: INFO: common: created directory at: artifacts/model_evaluation]
[2026-09-28 11:48:28,129: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


2026/09/28 11:48:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'ElasticNetModel'.
2026/09/28 11:48:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNetModel, version 1
Created version '1' of model 'ElasticNetModel'.


🏃 View run unruly-shoat-112 at: https://dagshub.com/pervnandal/datascienceproject.mlflow/#/experiments/0/runs/5b05efd132c043389626a6728e1f1e9d
🧪 View experiment at: https://dagshub.com/pervnandal/datascienceproject.mlflow/#/experiments/0
